# Customer Churn: Data Quality Audit & Exploratory Analysis
**Purpose:** audit and clean the supplied synthetic customer churn dataset, then inspect initial patterns.

> **Provenance note:** The source records were transcribed from the dataset screenshot supplied in the task conversation. The screenshot shows 15 records. Before final submission, compare this transcription with the downloadable original CSV and replace the input file if it contains additional records or differs in any value.

## Decision question
Which customer groups should a fictional subscription team contact first to reduce avoidable churn?

## Dataset fields
CustomerID, Gender, Age, TenureMonths, SubscriptionType, MonthlyCharges, TotalCharges, ContractType, SupportTickets, PaymentMethod, Churn.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Keep the raw file unchanged. Put customer_churn_raw_transcribed.csv
# in the same folder as this notebook before running.
df = pd.read_csv("customer_churn_raw_transcribed.csv")
df.head()


## 1. Profile columns and data types

In [ ]:
print("Shape:", df.shape)
display(df.dtypes)
display(df.describe(include="all").T)


## 2. Missing values, duplicates and unusual values

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique()
})
display(quality)
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate CustomerID values:", df["CustomerID"].duplicated().sum())


## 3. Cleaning decisions
- Trim whitespace from text fields and standardize text storage.
- Convert age, tenure and ticket counts to nullable integers.
- Convert monetary columns to numeric values rounded to two decimal places.
- Do not impute or delete records: this 15-row screenshot transcription contains no missing values or duplicate IDs/rows.
- Do not automatically remove statistical outliers in this tiny dataset. Investigate them with domain context first.
- Validate allowed categories, non-negative measures, unique IDs and the arithmetic relationship `TotalCharges ≈ MonthlyCharges × TenureMonths`.


In [ ]:
clean = df.copy()
for col in ["Age", "TenureMonths", "SupportTickets"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce").astype("Int64")
for col in ["MonthlyCharges", "TotalCharges"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce").round(2)
for col in ["CustomerID", "Gender", "SubscriptionType", "ContractType", "PaymentMethod", "Churn"]:
    clean[col] = clean[col].astype("string").str.strip()

checks = {
    "CustomerID unique": clean["CustomerID"].is_unique,
    "Age positive": (clean["Age"] > 0).all(),
    "Tenure non-negative": (clean["TenureMonths"] >= 0).all(),
    "Charges non-negative": (clean[["MonthlyCharges", "TotalCharges"]] >= 0).all().all(),
    "Support tickets non-negative": (clean["SupportTickets"] >= 0).all(),
    "Churn contains only Yes/No": clean["Churn"].isin(["Yes", "No"]).all(),
    "TotalCharges ~= MonthlyCharges * TenureMonths": np.allclose(
        clean["TotalCharges"],
        clean["MonthlyCharges"] * clean["TenureMonths"],
        atol=0.011
    )
}
pd.Series(checks, name="passed")


## 4. Summary statistics and churn patterns

In [ ]:
display(clean.describe(include="all").T)
print("Overall churn rate:", round((clean["Churn"].eq("Yes").mean() * 100), 2), "%")
display(clean.groupby("ContractType")["Churn"].agg(
    customers="size",
    churned=lambda s: s.eq("Yes").sum(),
    churn_rate=lambda s: s.eq("Yes").mean() * 100
).round(2))
display(clean.groupby("SubscriptionType")["Churn"].agg(
    customers="size",
    churned=lambda s: s.eq("Yes").sum(),
    churn_rate=lambda s: s.eq("Yes").mean() * 100
).round(2))


In [ ]:
clean["Churn"].value_counts().reindex(["Yes", "No"]).plot(kind="bar", title="Customer churn counts")
plt.xlabel("Churn"); plt.ylabel("Customers"); plt.tight_layout(); plt.show()

(clean.groupby("ContractType")["Churn"]
 .apply(lambda s: s.eq("Yes").mean() * 100)
 .sort_values(ascending=False)
 .plot(kind="bar", title="Churn rate by contract type"))
plt.xlabel("Contract type"); plt.ylabel("Churn rate (%)"); plt.tight_layout(); plt.show()


## 5. Export and reproducibility
The cleaned output is written without changing the raw input. Keep the notebook, raw CSV, cleaned CSV and report together in one repository/folder.

```python
clean.to_csv("cleaned_customer_churn.csv", index=False)
```

## 6. Interpretation and limitations
This is a tiny synthetic sample (15 customers), so percentages can change sharply with one customer. The patterns are descriptive, not causal and not predictive. Do not generalize them to a real customer base. Validate against the original downloadable dataset before publishing.
